In [ ]:
!pip -q install gradio transformers torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [36]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    AutoModelForSequenceClassification,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

In [41]:
act_model_path = "/content/drive/MyDrive/DS 266/act_model"
emotion_model_path = "/content/drive/MyDrive/DS 266/emotion_model"

act_tokenizer = AutoTokenizer.from_pretrained(act_model_path)
act_model = AutoModelForSequenceClassification.from_pretrained(act_model_path).to(device)
emotion_tokenizer = AutoTokenizer.from_pretrained(emotion_model_path)
emotion_model = AutoModelForSequenceClassification.from_pretrained(emotion_model_path).to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [42]:
translator_path = '/content/drive/MyDrive/DS 266/final_bart_genz_model'

translator_tokenizer = AutoTokenizer.from_pretrained(translator_path)
translator_model = AutoModelForSeq2SeqLM.from_pretrained(translator_path).to(device)
translator_model.eval()


Loading weights:   0%|          | 0/260 [00:01<?, ?it/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [43]:
max_source_length = 96
max_target_length = 64

In [44]:
def predict_label(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1)

    return torch.argmax(probs).item()

In [45]:
def predict_probs(text, tokenizer, model):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1).squeeze(0)

    pred_label = int(torch.argmax(probs).item())
    return pred_label, probs.cpu()

In [78]:
def generate_candidates(text, num_return_sequences=5, do_sample = False, direction="standard_to_genz", max_new_tokens=50, num_beams=8):
    if direction == "standard_to_genz":
        prompt = f"translate to Gen Z: {text}"
    elif direction == "genz_to_standard":
        prompt = f"translate to standard English: {text}"
    else:
        raise ValueError("direction must be 'standard_to_genz' or 'genz_to_standard'")

    inputs = translator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = translator_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True,
            num_return_sequences=num_return_sequences

        )

    decoded = [
      translator_tokenizer.decode(output, skip_special_tokens=True).strip()
      for output in outputs
    ]

    # remove exact duplicates while preserving order
    unique_candidates = list(dict.fromkeys(decoded))

    return unique_candidates

In [79]:
def score_candidate(original_text, candidate_text, alpha=1.0, beta=1.0):
    # Original predictions
    orig_act_label, orig_act_probs = predict_probs(original_text, act_tokenizer, act_model)
    orig_emotion_label, orig_emotion_probs = predict_probs(original_text, emotion_tokenizer, emotion_model)

    # Candidate predictions
    cand_act_label, cand_act_probs = predict_probs(candidate_text, act_tokenizer, act_model)
    cand_emotion_label, cand_emotion_probs = predict_probs(candidate_text, emotion_tokenizer, emotion_model)

    # Reward matching predicted labels
    act_match = 1.0 if cand_act_label == orig_act_label else 0.0
    emotion_match = 1.0 if cand_emotion_label == orig_emotion_label else 0.0

    # Also reward confidence on the original predicted class
    act_preservation = float(cand_act_probs[orig_act_label].item())
    emotion_preservation = float(cand_emotion_probs[orig_emotion_label].item())

    # Final score
    score = (
        alpha * (0.5 * act_match + 0.5 * act_preservation) +
        beta * (0.5 * emotion_match + 0.5 * emotion_preservation)
    )

    return {
        "candidate": candidate_text,
        "score": score,
        "orig_act": orig_act_label,
        "cand_act": cand_act_label,
        "orig_emotion": orig_emotion_label,
        "cand_emotion": cand_emotion_label,
        "act_preservation": act_preservation,
        "emotion_preservation": emotion_preservation
    }

In [80]:
def translate_to_genz_with_guardrail(text, num_candidates=5):
    candidates = generate_candidates(text, num_return_sequences=num_candidates)
    print(candidates)

    scored = [score_candidate(text, c) for c in candidates]
    scored = sorted(scored, key=lambda x: x["score"], reverse=True)

    best = scored[0]
    print(best["candidate"])
    return(best["candidate"])

In [82]:
import gradio as gr


demo = gr.Interface(
    fn=translate_to_genz_with_guardrail,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Type your sentence here..."
    ),
    outputs=gr.Textbox(label="Gen Z Translation"),
    title="English → Gen Z Translator",
    description="Enter a sentence and get a Gen Z-style rewrite"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2f381c588204575b70.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
